In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

import numpy as np
import pandas as pd
from tqdm import tqdm
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import os
from PIL import Image
from collections import Counter
import glob
import chess
import random
# pandas display settings (kept from your original snippet)
pd.set_option('display.max_columns', None)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x 
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual 
        out = F.relu(out)
        return out

# 2. The DUAL-HEAD Chess Neural Network
class ChessResNet(nn.Module):
    def __init__(self, num_blocks=10, hidden_channels=128):
        super(ChessResNet, self).__init__()
        
        self.start_conv = nn.Conv2d(12, hidden_channels, kernel_size=3, padding=1)
        self.start_bn = nn.BatchNorm2d(hidden_channels)
        self.res_blocks = nn.ModuleList([ResBlock(hidden_channels) for _ in range(num_blocks)])
        
        # --- FIXED POLICY HEAD (76 Planes) ---
        self.policy_conv = nn.Conv2d(hidden_channels, 76, kernel_size=1) 
        self.policy_bn = nn.BatchNorm2d(76)
        
        # 76 planes * 8 * 8 squares = 4864 nodes
        self.policy_fc = nn.Linear(76 * 8 * 8, 4864) 
        
        # --- Value Head ---
        self.eval_conv = nn.Conv2d(hidden_channels, 1, kernel_size=1)
        self.eval_bn = nn.BatchNorm2d(1)
        self.eval_fc1 = nn.Linear(8 * 8, 64) 
        self.eval_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.start_bn(self.start_conv(x)))
        for block in self.res_blocks:
            x = block(x)
            
        # Policy output
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.view(-1, 4864) # Flatten to the correct 4864 size
        policy_out = self.policy_fc(p) 
        
        # Value output
        v = F.relu(self.eval_bn(self.eval_conv(x)))
        v = v.view(-1, 64) 
        v = F.relu(self.eval_fc1(v))
        v = self.eval_fc2(v)
        value_out = torch.tanh(v) 
        
        return policy_out, value_out
model = ChessResNet(num_blocks=10, hidden_channels=128).to(device)
model.load_state_dict(torch.load(
    r'Models\chess_model_v3.pth',
    map_location=device,
    weights_only=True
))
model.eval()

ChessResNet(
  (start_conv): Conv2d(12, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (start_bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (res_blocks): ModuleList(
    (0-9): 10 x ResBlock(
      (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (policy_conv): Conv2d(128, 76, kernel_size=(1, 1), stride=(1, 1))
  (policy_bn): BatchNorm2d(76, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (policy_fc): Linear(in_features=4864, out_features=4864, bias=True)
  (eval_conv): Conv2d(128, 1, kernel_size=(1, 1), stride=(1, 1))
  (eval_bn): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
 

In [6]:
import torch
import torch.onnx

model = ChessResNet()
model.eval()

# 1. Create your dummy input
dummy_input = torch.randn(1, 12, 8, 8)

# 2. Define the dynamic shape for the batch dimension
# We use a 'dim' object to tell PyTorch the first dimension (index 0) is dynamic
batch = torch.export.Dim("batch", min=1, max=1024)
dynamic_shapes = {"x": {0: batch}}

# 3. Export using the updated API
onnx_program = torch.onnx.export(
    model,
    (dummy_input,),
    "chess_resnet.onnx",
    input_names=['x'],
    output_names=['policy', 'value'],
    dynamic_shapes=dynamic_shapes, # The new way to handle it
    opset_version=17 # Higher opset usually handles modern layers better
)

print("Export complete with dynamic shapes!")

W0326 08:42:38.727000 33332 site-packages\torch\onnx\_internal\exporter\_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0326 08:42:38.988000 33332 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0326 08:42:38.989000 33332 site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_sc

[torch.onnx] Obtain model graph for `ChessResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ChessResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 23 of general pattern rewrite rules.
Export complete with dynamic shapes!


In [7]:
import onnxruntime as ort

# Create a session options object
sess_options = ort.SessionOptions()

# 1. Enable ALL graph optimizations (Node fusion, constant folding, etc.)
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# 2. Optimize CPU threading (tune this to your CPU's physical cores)
sess_options.intra_op_num_threads = 4 

# Load the session with these options
session = ort.InferenceSession("chess_resnet.onnx", sess_options)

In [11]:
# The order matters! It will try TensorRT first, then CUDA, then fallback to CPU
providers = [
    'CUDAExecutionProvider',
    'CPUExecutionProvider'
]

session = ort.InferenceSession("chess_resnet.onnx", sess_options, providers=providers)

In [13]:
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process

# File paths
model_fp32 = "chess_resnet.onnx"
model_prep = "chess_resnet_prepped.onnx"
model_int8 = "chess_resnet_int8.onnx"

print("1. Running pre-processing and shape inference...")
# This cleans the graph and saves a new, optimized FP32 model
quant_pre_process(
    input_model_path=model_fp32,
    output_model_path=model_prep,
    skip_optimization=False
)

print("2. Quantizing the pre-processed model...")
# Now we quantize the CLEANED model, not the raw PyTorch export
quantize_dynamic(
    model_input=model_prep,
    model_output=model_int8,
    weight_type=QuantType.QUInt8
)

print("Done! 'chess_resnet_int8.onnx' is fully optimized and ready.")

1. Running pre-processing and shape inference...
2. Quantizing the pre-processed model...
Done! 'chess_resnet_int8.onnx' is fully optimized and ready.


In [59]:
batch_size = 1
dummy_board = np.random.randn(batch_size, 12, 8, 8)

In [64]:
import numpy as np
import onnxruntime as ort
import time

# 1. Load the FP32 (or FP16) Model onto the GPU
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

print("Loading model to GPU...")
session = ort.InferenceSession(
    "chess_resnet_int8.onnx", # The clean floating-point model
    session_options, 
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

# Verify we are actually using CUDA
active_providers = session.get_providers()
print(f"Active Execution Providers: {active_providers}")
if 'CUDAExecutionProvider' not in active_providers:
    print("WARNING: CUDA failed to load. Falling back to CPU.")

# 2. Get the input and output names dynamically
input_name = session.get_inputs()[0].name
policy_name = session.get_outputs()[0].name
value_name = session.get_outputs()[1].name

# 3. Create the Dummy Chess Board
batch_size = 1
dummy_board = np.random.randn(batch_size, 12, 8, 8).astype(np.float32)

# --- THE BENCHMARK ---
print("\nRunning warm-up (GPUs need a second to wake up)...")
for _ in range(50):
    session.run([policy_name, value_name], {input_name: dummy_board})

print("Running benchmark (1,000 inferences)...")
runs = 1000
start_time = time.perf_counter()

for _ in range(runs):
    outputs = session.run([policy_name, value_name], {input_name: dummy_board})

end_time = time.perf_counter()
total_time = end_time - start_time
time_per_run = (total_time / runs) * 1000 # Convert to milliseconds

# 4. Extract the final results
policy_out = outputs[0] 
value_out = outputs[1]  

print("\n--- INFERENCE SUCCESS ---")
print(f"Policy Shape: {policy_out.shape} | First 5 move logits: {policy_out[0][:5]}")
print(f"Value Shape: {value_out.shape}   | Position Evaluation: {value_out[0][0]:.3f}")

print("\n--- SPEED REPORT ---")
print(f"Total time for {runs} boards: {total_time:.3f} seconds")
print(f"Time per board: {time_per_run:.3f} milliseconds")
print(f"Speed: {int(runs / total_time):,} boards per second (at Batch Size 1)")

Loading model to GPU...
Active Execution Providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Running warm-up (GPUs need a second to wake up)...
Running benchmark (1,000 inferences)...

--- INFERENCE SUCCESS ---
Policy Shape: (1, 4864) | First 5 move logits: [-0.1415388   0.01150575 -0.01333407  0.03364909  0.10067328]
Value Shape: (1, 1)   | Position Evaluation: -0.012

--- SPEED REPORT ---
Total time for 1000 boards: 5.690 seconds
Time per board: 5.690 milliseconds
Speed: 175 boards per second (at Batch Size 1)


In [69]:
import numpy as np
import onnxruntime as ort
import time

# 1. Load the FP32 (or FP16) Model onto the GPU
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

print("Loading model to GPU...")
session = ort.InferenceSession(
    "chess_resnet_prepped.onnx", # The clean floating-point model
    session_options, 
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

# Verify we are actually using CUDA
active_providers = session.get_providers()
print(f"Active Execution Providers: {active_providers}")
if 'CUDAExecutionProvider' not in active_providers:
    print("WARNING: CUDA failed to load. Falling back to CPU.")

# 2. Get the input and output names dynamically
input_name = session.get_inputs()[0].name
policy_name = session.get_outputs()[0].name
value_name = session.get_outputs()[1].name

# 3. Create the Dummy Chess Board
batch_size = 1
dummy_board = np.random.randn(batch_size, 12, 8, 8).astype(np.float32)

# --- THE BENCHMARK ---
print("\nRunning warm-up (GPUs need a second to wake up)...")
for _ in range(50):
    session.run([policy_name, value_name], {input_name: dummy_board})

print("Running benchmark (1,000 inferences)...")
runs = 1000
start_time = time.perf_counter()

for _ in range(runs):
    outputs = session.run([policy_name, value_name], {input_name: dummy_board})

end_time = time.perf_counter()
total_time = end_time - start_time
time_per_run = (total_time / runs) * 1000 # Convert to milliseconds

# 4. Extract the final results
policy_out = outputs[0] 
value_out = outputs[1]  

print("\n--- INFERENCE SUCCESS ---")
print(f"Policy Shape: {policy_out.shape} | First 5 move logits: {policy_out[0][:5]}")
print(f"Value Shape: {value_out.shape}   | Position Evaluation: {value_out[0][0]:.3f}")

print("\n--- SPEED REPORT ---")
print(f"Total time for {runs} boards: {total_time:.3f} seconds")
print(f"Time per board: {time_per_run:.3f} milliseconds")
print(f"Speed: {int(runs / total_time):,} boards per second (at Batch Size 1)")

Loading model to GPU...
Active Execution Providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

Running warm-up (GPUs need a second to wake up)...
Running benchmark (1,000 inferences)...

--- INFERENCE SUCCESS ---
Policy Shape: (1, 4864) | First 5 move logits: [-0.06792572 -0.01095598  0.08033767 -0.01639152  0.10193729]
Value Shape: (1, 1)   | Position Evaluation: -0.002

--- SPEED REPORT ---
Total time for 1000 boards: 1.912 seconds
Time per board: 1.912 milliseconds
Speed: 522 boards per second (at Batch Size 1)


In [70]:
model = torch.compile(model)

In [71]:
import torch
import time
import numpy as np

# 1. Setup Model and Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessResNet().to(device) # CRITICAL: Move model to GPU
model.eval()

# 2. Create Dummy Data (Match your ONNX batch size for a fair fight)
batch_size = 1 
dummy_board_np = np.random.randn(batch_size, 12, 8, 8).astype(np.float32)
# Convert to tensor and move to GPU
x = torch.from_numpy(dummy_board_np).to(device)

# 3. Benchmark
print(f"Running PyTorch benchmark on {device}...")
print("Warming up...")
with torch.inference_mode():
    for _ in range(50):
        _ = model(x)

    print(f"Inference starting (1,000 runs, batch_size={batch_size})...")
    runs = 1000
    start_time = time.perf_counter()

    for _ in range(runs):
        policy, value = model(x)
        # Optional: torch.cuda.synchronize() 
        # (Use synchronize if you want to be extremely precise about GPU timing)

    end_time = time.perf_counter()

# 4. Results
total_time = end_time - start_time
time_per_run = (total_time / runs) * 1000

print("\n--- PYTORCH SPEED REPORT ---")
print(f"Total time: {total_time:.3f} seconds")
print(f"Time per board: {time_per_run:.3f} milliseconds")
print(f"Speed: {int((runs * batch_size) / total_time):,} boards per second")

Running PyTorch benchmark on cuda...
Warming up...
Inference starting (1,000 runs, batch_size=1)...

--- PYTORCH SPEED REPORT ---
Total time: 2.837 seconds
Time per board: 2.837 milliseconds
Speed: 352 boards per second


In [79]:
import numpy as np
import onnxruntime as ort
import torch

# 1. Load the GPU model (Prepped FP32 or FP16)
session = ort.InferenceSession("chess_resnet_prepped.onnx", 
                               providers=['CUDAExecutionProvider'])

# 2. Get metadata
input_name = session.get_inputs()[0].name
input_shape = session.get_inputs()[0].shape # e.g., ['batch', 12, 8, 8]
# Since 'batch' is dynamic, we'll pick a fixed test batch size
batch_sz = 64 

# 3. Pre-allocate memory ON THE GPU using PyTorch (easiest way)
# We create tensors directly on the GPU so ONNX doesn't have to copy them
x_gpu = torch.randn(batch_sz, 12, 8, 8, device='cuda', dtype=torch.float32)
p_out_gpu = torch.empty(batch_sz, 4864, device='cuda', dtype=torch.float32)
v_out_gpu = torch.empty(batch_sz, 1, device='cuda', dtype=torch.float32)

# 4. Create the Binding object
io_binding = session.io_binding()

# Bind the Input
io_binding.bind_input(
    name=input_name,
    device_type='cuda',
    device_id=0,
    element_type=np.float32,
    shape=(batch_sz, 12, 8, 8),
    buffer_ptr=x_gpu.data_ptr() # Point directly to the GPU memory address
)

# Bind the Outputs
io_binding.bind_output(
    name='policy',
    device_type='cuda',
    device_id=0,
    element_type=np.float32,
    shape=(batch_sz, 4864),
    buffer_ptr=p_out_gpu.data_ptr()
)

io_binding.bind_output(
    name='value',
    device_type='cuda',
    device_id=0,
    element_type=np.float32,
    shape=(batch_sz, 1),
    buffer_ptr=v_out_gpu.data_ptr()
)

# 5. Run Inference (Zero-Copy!)
# Notice we don't pass a dictionary or get a return value here
session.run_with_iobinding(io_binding)

# The results are already inside p_out_gpu and v_out_gpu!
print(f"Policy Sample: {p_out_gpu[0][:5]}")

Policy Sample: tensor([-0.0821,  0.1303,  0.0330,  0.0501,  0.0434], device='cuda:0')
